In [4]:
import os; os.chdir("..")

In [5]:
from app.tampering.dataset_builder import build_dataset, validate_dataset
r = build_dataset("passport", "data/tampering_dataset")
print(r["success"], r["errors"])
print(validate_dataset("data/tampering_dataset"))

False ['Source directory not found: passport']
{'valid': False, 'genuine_count': 0, 'tampered_count': 0, 'missing_files': [], 'invalid_labels': [], 'duplicates': [], 'errors': ['Dataset root does not exist: data\\tampering_dataset'], 'total': 0}


In [12]:
import pandas as pd, random
from pathlib import Path
from app.tampering.train import train

root = Path("prototype/data/tampering_dataset")
df = pd.read_csv("prototype/data/tampering_dataset/manifest.csv")
df["image_path"] = df["image_path"].apply(lambda p: str(root / p.replace("\\", "/")))

ids = sorted(df["base_document_id"].unique())
random.Random(42).shuffle(ids)
test_ids, val_ids, train_ids = ids[:3], ids[3:5], ids[5:]   # split by passport, no leakage
print("train:", train_ids, "\nval:", val_ids, "\ntest:", test_ids)

def mk(id_list):
    d = df[df["base_document_id"].isin(id_list)]
    return [{"image_path": r.image_path, "label": int(r.label), "base_document_id": r.base_document_id}
            for r in d.itertuples()]

out = train(mk(train_ids), mk(val_ids), "data/model_checkpoints/efficientnet_b0",
            freeze_backbone=True, unfreeze_after=2, lr=3e-4, batch_size=16,
            epochs=12, patience=4, use_weighted_sampler=False)
print(out["best_val_acc"], out["best_epoch"], out["errors"])

train: ['passport2', 'passport8', 'passport5', 'passport12', 'passport13', 'passport1', 'passport10', 'passport7'] 
val: ['passport6', 'passport9'] 
test: ['passport4', 'passport3', 'passport11']
Epoch   1/12  train_loss=0.6862  train_acc=0.5664  val_loss=0.6874  val_acc=0.5750  (40.3s)
  ✓ New best val_acc=0.5750 — checkpoint saved.
Epoch   2/12  train_loss=0.6458  train_acc=0.6328  val_loss=0.6480  val_acc=0.6125  (92.1s)
  ✓ New best val_acc=0.6125 — checkpoint saved.
Epoch   3/12  train_loss=0.5842  train_acc=0.7383  val_loss=0.6021  val_acc=0.6875  (92.0s)
  ✓ New best val_acc=0.6875 — checkpoint saved.
Epoch   4/12  train_loss=0.5247  train_acc=0.7891  val_loss=0.5385  val_acc=0.7312  (89.7s)
  ✓ New best val_acc=0.7312 — checkpoint saved.
Epoch   5/12  train_loss=0.4574  train_acc=0.8008  val_loss=0.5044  val_acc=0.7500  (90.4s)
  ✓ New best val_acc=0.7500 — checkpoint saved.
Epoch   6/12  train_loss=0.3922  train_acc=0.8711  val_loss=0.4772  val_acc=0.7625  (90.0s)
  ✓ New best

In [14]:
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix
from app.tampering.inference import predict

CK = "prototype/data/model_checkpoints/efficientnet_b0/best_checkpoint.pt"
rows = []
for s in mk(test_ids):
    r = predict(s["image_path"], CK)
    rows.append((Path(s["image_path"]).name, s["label"], r["score_tampered"], r["band"]))
for name, y, sc, band in sorted(rows, key=lambda x: x[1]):
    print(f"{'GENUINE ' if y==0 else 'TAMPERED'}  score={sc:.3f}  band={band:9s}  {name}")
y = np.array([x[1] for x in rows]); sc = np.array([x[2] for x in rows])
print("AUC:", round(roc_auc_score(y, sc), 3))
print(confusion_matrix(y, (sc > 0.5).astype(int)))

# real-world check: your phone photo of the tampered passport
print(predict(r"PATH\TO\T_photo_test.jpg", CK)["band"])

GENUINE   score=0.513  band=UNCERTAIN  passport11_genuine_original.jpg
GENUINE   score=0.606  band=UNCERTAIN  passport3_genuine_original.jpg
GENUINE   score=0.584  band=UNCERTAIN  passport4_genuine_original.jpg
TAMPERED  score=0.512  band=UNCERTAIN  passport11_tampered_photo_repl_42.jpg
TAMPERED  score=0.533  band=UNCERTAIN  passport11_tampered_text_dob_42.jpg
TAMPERED  score=0.527  band=UNCERTAIN  passport11_tampered_copypaste_42.jpg
TAMPERED  score=0.513  band=UNCERTAIN  passport11_tampered_colourshift_42.jpg
TAMPERED  score=0.525  band=UNCERTAIN  passport11_tampered_pen_dob_42.jpg
TAMPERED  score=0.519  band=UNCERTAIN  passport11_tampered_pen_number_42.jpg
TAMPERED  score=0.503  band=UNCERTAIN  passport11_tampered_mrz_scribble_42.jpg
TAMPERED  score=0.632  band=UNCERTAIN  passport3_tampered_photo_repl_42.jpg
TAMPERED  score=0.598  band=UNCERTAIN  passport3_tampered_text_dob_42.jpg
TAMPERED  score=0.592  band=UNCERTAIN  passport3_tampered_copypaste_42.jpg
TAMPERED  score=0.604  band=